# X(독립변수) 조합 선정 검토 — 오준영님 조합 A·B 를 다시 재 본다

**작성** 이동원 (데이터 파트) · **2026-09-01**

오준영님이 조합 A·B 로지스틱 실험 결과를 공유하며 *"기준과 가설이 적절한지,
어떤 피처를 쓰면 좋을지"* 검토를 요청하셨습니다.

이 노트북은 **먼저 그 결과를 그대로 재현하고**, 그다음에 무엇이 문제였는지 잽니다.
재현되지 않는 지적은 오해일 때가 많기 때문입니다.

결론 문서 → [`docs/모델파트/version1.0/X조합-피처선정-검토.md`](../../docs/모델파트/version1.0/X조합-피처선정-검토.md)

| 순서 | 무엇을 하나 |
|---|---|
| 1 | 자료를 받는다 |
| 2 | **조합 A·B 를 준영님 설정 그대로 재현한다** |
| 3 | 기준 ④ 중복 — 정의상 같은 열이 있는지 |
| 4 | 기준 ⑤ 정상성 — 학습 범위 밖으로 나가는 비율 |
| 5 | 기준 ①②③ — 어느 피처가 어느 질문에 답하나 |
| 6 | 문제 분해 — 3분류는 크기 문제 + 방향 문제다 |
| 7 | 조합 6안 × 모델 4종 성적표 |
| 8 | 하락 예측이 안 나온 진짜 이유 |
| 9 | 유효표본과 유의선 |


## 1. 자료를 받는다

팀 비공개 HuggingFace 데이터셋에서 받습니다(준영님 노트북과 같은 경로).
토큰이 없거나 받지 못하면 로컬 `data/outbox/` 로 물러납니다 —
**둘 다 같은 파일이고 SHA-256 이 `MANIFEST.json` 에 적혀 있습니다.**

In [1]:
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

# 프로젝트 루트를 찾아 sys.path 에 넣는다 (evaluation·models 를 쓰기 위해)
project_root = Path.cwd().resolve()
while project_root != project_root.parent and not (project_root / "models").is_dir():
    project_root = project_root.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

REL = "small/features_labels_kospi200_dev.csv"


def load_dataset():
    """HuggingFace 를 먼저 보고, 안 되면 로컬 outbox 로 물러난다."""
    try:
        from huggingface_hub import hf_hub_download

        from common import secrets

        token, source = secrets.load_key(("HUGGINGFACE_ACCESS_TOKEN",))
        if token and token.startswith("hf_") and token.isascii():
            path = hf_hub_download(
                repo_id="qurious-quant/alphastack-krx-dev",
                filename=REL,
                repo_type="dataset",
                token=token,
            )
            return path, f"HuggingFace (토큰 출처: {source})"
    except Exception as error:  # noqa: BLE001 — 어떤 이유든 로컬로 물러난다
        print("HuggingFace 경로 실패 → 로컬로 물러납니다:", type(error).__name__)

    local = project_root / "data" / "outbox" / "2026-08-31" / REL
    if not local.exists():
        raise FileNotFoundError(f"자료를 찾지 못했습니다: {local}")
    return str(local), "로컬 data/outbox"


data_path, source = load_dataset()
df = pd.read_csv(data_path, dtype={"bas_dd": "string"})
df = df.sort_values("bas_dd").reset_index(drop=True)
df["y"] = df["label"].str.strip().map({"하락": -1, "중립": 0, "상승": 1}).astype(int)

print("자료 출처:", source)
print("크기:", df.shape, "| 기간:", df["bas_dd"].iloc[0], "~", df["bas_dd"].iloc[-1])
print("\n전체 라벨 분포(%)")
print((df["y"].value_counts(normalize=True).sort_index() * 100).round(2).to_string())

자료 출처: HuggingFace (토큰 출처: .env)
크기: (2815, 38) | 기간: 20100330 ~ 20210823

전체 라벨 분포(%)
y
-1    27.32
 0    38.69
 1    34.00


## 2. 조합 A·B 를 준영님 설정 그대로 재현한다

준영님 노트북(`notebooks/04-모델/실험/`)의 설정입니다.

```python
expanding_splits(n_samples=2815, n_folds=12, min_train=750,
                 horizon=60, gap=5, label_horizon=5)
```

조합 B 의 `sma_gap_5_20`·`macd_hist_ratio` 는 준영님이 직접 만드신 파생 피처라
같은 정의로 다시 만듭니다.

In [2]:
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score

from evaluation.walk_forward import expanding_splits
from models.logistic import build_logistic_baseline

# 준영님이 만드신 파생 피처 (같은 정의)
df["sma_gap_5_20"] = df["sma_5"] / df["sma_20"] - 1.0
df["macd_hist_ratio"] = df["macd_hist"] / df["close"]

COMBO_A = ["rsi_14", "bb_bandwidth", "hv_20", "vol_ratio_20"]
COMBO_B = ["sma_gap_5_20", "macd_hist_ratio", "rsi_14", "hv_20"]

n = len(df)
splits = expanding_splits(
    n_samples=n, n_folds=12, min_train=750, horizon=60, gap=5, label_horizon=5,
)
valid_index = np.concatenate([v for _, v in splits])
y_valid_all = df.loc[valid_index, "y"].to_numpy()

print("폴드:", len(splits), "| 폴드별 검증:", len(splits[0][1]),
      "| 검증 합계:", len(valid_index))
print("검증 기간:", df.loc[valid_index[0], "bas_dd"], "~",
      df.loc[valid_index[-1], "bas_dd"])
print("\n검증 구간 분포")
for name, value in (("하락", -1), ("중립", 0), ("상승", 1)):
    mask = y_valid_all == value
    print(f"  {name}: {mask.sum():3d}개 ({mask.mean() * 100:.2f}%)")

폴드: 12 | 폴드별 검증: 60 | 검증 합계: 720
검증 기간: 20130409 ~ 20210823

검증 구간 분포
  하락: 204개 (28.33%)
  중립: 306개 (42.50%)
  상승: 210개 (29.17%)


In [3]:
def walk_forward(feature_columns, build=build_logistic_baseline):
    """폴드마다 새 모델을 학습해 검증 구간을 예측한다. 인덱스와 예측을 돌려준다."""
    kept_index, predictions = [], []
    for train_index, valid_idx in splits:
        x_train = df.loc[train_index, feature_columns].to_numpy(float)
        y_train = df.loc[train_index, "y"].to_numpy()
        usable = np.isfinite(x_train).all(axis=1)

        model = build()
        model.fit(x_train[usable], y_train[usable])

        x_valid = df.loc[valid_idx, feature_columns].to_numpy(float)
        usable_valid = np.isfinite(x_valid).all(axis=1)
        kept_index.append(np.asarray(valid_idx)[usable_valid])
        predictions.append(np.asarray(model.predict(x_valid[usable_valid])).astype(int))
    return np.concatenate(kept_index), np.concatenate(predictions)


def score(tag, index, prediction):
    """정확도·Macro F1·클래스별 Recall 을 한 줄로 인쇄한다."""
    truth = df.loc[index, "y"].to_numpy()
    matrix = confusion_matrix(truth, prediction, labels=[-1, 0, 1])
    recall = [matrix[i, i] / max(1, matrix[i].sum()) * 100 for i in range(3)]
    print(
        f"{tag:24s} Acc={accuracy_score(truth, prediction):.4f} "
        f"MacroF1={f1_score(truth, prediction, average='macro'):.4f} "
        f"Recall(하/중/상)={recall[0]:5.1f}/{recall[1]:5.1f}/{recall[2]:5.1f}% "
        f"하락예측={int((prediction == -1).sum()):3d}"
    )
    return matrix


print("준영님 보고: 조합 A Acc=0.3958 MacroF1=0.2971 하락예측 0개")
print("             조합 B Acc=0.3792 MacroF1=0.2965 하락 Recall 1.96%")
print("-" * 78)
matrix_a = score("조합 A (재현)", *walk_forward(COMBO_A))
matrix_b = score("조합 B (재현)", *walk_forward(COMBO_B))
print("\n조합 A 혼동행렬 (행=실제 하락/중립/상승, 열=예측)")
print(pd.DataFrame(matrix_a, index=["하락", "중립", "상승"],
                   columns=["→하락", "→중립", "→상승"]).to_string())

준영님 보고: 조합 A Acc=0.3958 MacroF1=0.2971 하락예측 0개
             조합 B Acc=0.3792 MacroF1=0.2965 하락 Recall 1.96%
------------------------------------------------------------------------------
조합 A (재현)                Acc=0.3958 MacroF1=0.2971 Recall(하/중/상)=  0.0/ 63.4/ 43.3% 하락예측=  0


조합 B (재현)                Acc=0.3792 MacroF1=0.2965 Recall(하/중/상)=  2.0/ 58.5/ 42.9% 하락예측=  8

조합 A 혼동행렬 (행=실제 하락/중립/상승, 열=예측)
    →하락  →중립  →상승
하락    0  125   79
중립    0  194  112
상승    0  119   91


✅ **소수점까지 일치합니다.** 이제 같은 무대 위에서 이야기할 수 있습니다.

조합 A 의 혼동행렬 첫 열이 전부 0 입니다 — 하락을 **한 번도 부르지 않았습니다.**

## 3. 기준 ④ 중복 — 정의상 **같은 열**이 있는지

준영님 기준 중 *"같은 내용을 나타내는 비슷한 피처는 중복해서 넣지 않기"* 를
자료로 확인합니다. 비슷한 정도가 아니라 **수학적으로 같은 열**이 있습니다.

In [4]:
RAW22 = [
    "sma_5", "sma_20", "sma_60", "ema_12", "ema_26", "rsi_14",
    "macd", "macd_signal", "macd_hist", "bb_mid", "bb_upper", "bb_lower",
    "bb_bandwidth", "true_range", "atr_14", "hv_20", "parkinson_20",
    "vol_sma_20", "vol_ratio_20", "obv", "vwap_20", "vol_roc_5",
]

print("정의상 같은 열인지 — 값의 최대 차이로 확인")
print(f"  max|bb_mid - sma_20|            = {(df['bb_mid'] - df['sma_20']).abs().max():.3e}")
gap = (df["macd"] - (df["ema_12"] - df["ema_26"])).abs().max()
print(f"  max|macd - (ema_12 - ema_26)|   = {gap:.3e}")

print("\n상관이 높은 쌍 (피어슨)")
pairs = [
    ("bb_mid", "sma_20"), ("vwap_20", "sma_20"), ("ema_12", "ema_26"),
    ("sma_5", "sma_20"), ("sma_20", "sma_60"), ("macd", "macd_signal"),
    ("bb_upper", "bb_lower"), ("hv_20", "parkinson_20"), ("atr_14", "hv_20"),
]
for left, right in pairs:
    r = df[[left, right]].corr().iloc[0, 1]
    mark = "  ★ 사실상 같은 열" if abs(r) > 0.98 else ("  ⚠ 강한 중복" if abs(r) > 0.85 else "")
    print(f"  corr({left:13s}, {right:13s}) = {r:+.4f}{mark}")

정의상 같은 열인지 — 값의 최대 차이로 확인
  max|bb_mid - sma_20|            = 0.000e+00
  max|macd - (ema_12 - ema_26)|   = 1.141e-13

상관이 높은 쌍 (피어슨)
  corr(bb_mid       , sma_20       ) = +1.0000  ★ 사실상 같은 열
  corr(vwap_20      , sma_20       ) = +0.9999  ★ 사실상 같은 열
  corr(ema_12       , ema_26       ) = +0.9973  ★ 사실상 같은 열
  corr(sma_5        , sma_20       ) = +0.9918  ★ 사실상 같은 열
  corr(sma_20       , sma_60       ) = +0.9805  ★ 사실상 같은 열
  corr(macd         , macd_signal  ) = +0.9541  ⚠ 강한 중복
  corr(bb_upper     , bb_lower     ) = +0.9629  ⚠ 강한 중복
  corr(hv_20        , parkinson_20 ) = +0.9267  ⚠ 강한 중복
  corr(atr_14       , hv_20        ) = +0.8127


In [5]:
def variance_inflation(columns):
    """분산팽창계수(VIF). 10 을 넘으면 다중공선성 경고다."""
    matrix = df[columns].to_numpy(float)
    matrix = matrix[np.isfinite(matrix).all(axis=1)]
    matrix = (matrix - matrix.mean(0)) / matrix.std(0)
    result = {}
    for j, name in enumerate(columns):
        others = np.delete(matrix, j, axis=1)
        design = np.column_stack([np.ones(len(others)), others])
        beta, *_ = np.linalg.lstsq(design, matrix[:, j], rcond=None)
        r2 = 1 - (matrix[:, j] - design @ beta).var() / matrix[:, j].var()
        result[name] = 1 / max(1e-9, 1 - r2)
    return result


for tag, columns in (("조합 A", COMBO_A), ("조합 B", COMBO_B), ("원본 22개 전량", RAW22)):
    values = variance_inflation(columns)
    worst = max(values, key=values.get)
    print(f"  {tag:14s} 최대 VIF = {values[worst]:12.1f}  ({worst})")

  조합 A           최대 VIF =          2.2  (hv_20)
  조합 B           최대 VIF =          4.8  (sma_gap_5_20)
  원본 22개 전량      최대 VIF = 1000000000.0  (sma_20)


🔴 원본 22개를 전부 넣으면 VIF 가 **10⁹** 로 발산합니다.
`macd = ema_12 − ema_26` 이 **완전한 선형종속**이기 때문입니다.
규제가 예외를 막아 줄 뿐, 계수는 폴드마다 부호가 뒤집혀 해석이 불가능해집니다.

## 4. 기준 ⑤ 정상성 — 미래 누수는 없지만 **학습 범위 밖**이 남는다

`features/` 함수들은 과거→현재로만 누적하므로 누수는 없습니다.
문제는 다른 데 있습니다 — **가격·거래량 수준값**은 확장창에서
검증 구간이 학습이 본 적 없는 범위로 나갑니다.

In [6]:
rows = []
for column in RAW22:
    values = df[column].to_numpy(float)
    outside = total = 0
    for train_end in range(750, n - 60, 60):
        low, high = np.nanmin(values[:train_end]), np.nanmax(values[:train_end])
        window = values[train_end + 5: train_end + 65]
        window = window[np.isfinite(window)]
        outside += int(((window < low) | (window > high)).sum())
        total += len(window)
    half = n // 2
    rows.append({
        "피처": column,
        "전반부 평균": np.nanmean(values[:half]),
        "후반부 평균": np.nanmean(values[half:]),
        "학습범위 밖 %": outside / total * 100,
    })

table = pd.DataFrame(rows).sort_values("학습범위 밖 %", ascending=False)
print(table.to_string(index=False, float_format=lambda v: f"{v:12.4g}"))

          피처       전반부 평균       후반부 평균     학습범위 밖 %
         obv    3.322e+09    1.005e+10        21.91
      sma_60          252        295.7        16.81
      ema_26        252.4          298        13.19
      sma_20        252.5        298.4        12.99
      bb_mid        252.5        298.4        12.99
     vwap_20        252.5        298.4        12.94
    bb_lower          244        288.2        12.89
      ema_12        252.5        298.9        10.74
       sma_5        252.6        299.3        9.167
    bb_upper        260.9        308.5         7.99
  vol_sma_20    8.956e+07    1.186e+08        6.618
 macd_signal       0.1536       0.9147        3.775
      atr_14        3.341        3.953        3.088
        macd       0.1496       0.9026        2.892
parkinson_20     0.007312     0.007591        2.353
bb_bandwidth      0.06733      0.06865        1.569
       hv_20     0.009833     0.009631        1.569
   macd_hist    -0.004023     -0.01218       0.8824
      rsi_14

🔴 **트리 모델에서 특히 위험합니다.** 트리는 외삽을 못 하므로,
학습에서 배운 분할 임계값이 검증 구간 값을 전부 한쪽으로만 보냅니다.
준영님이 이 조합을 RandomForest·XGBoost·LightGBM 에 적용하실 계획이라
로지스틱보다 영향이 큽니다.

👍 준영님이 `sma_gap_5_20`·`macd_hist_ratio` 로 **비율화**하신 판단이
정확히 이 문제를 피한 것입니다.

## 5. 기준 ①②③ — 어느 피처가 **어느 질문에** 답하나

정상형 파생 피처를 만들고, 세 가지를 각각 잽니다.

- **IC** — 미래 5거래일 수익률과의 스피어만 상관 (방향 정보)
- **중립 판별 AUC** — `|5일수익률| > 1%` 를 가르는 능력 (크기 정보)
- **방향 판별 AUC** — 비중립 행 중 상승/하락을 가르는 능력

In [7]:
from scipy import stats

df["sma_gap_20_60"] = df["sma_20"] / df["sma_60"] - 1.0
df["px_vs_sma20"] = df["close"] / df["sma_20"] - 1.0
df["px_vs_sma60"] = df["close"] / df["sma_60"] - 1.0
df["vwap_gap"] = df["close"] / df["vwap_20"] - 1.0
df["macd_norm"] = df["macd"] / df["close"]
df["macd_hist_atr"] = df["macd_hist"] / df["atr_14"]
df["bb_pctb"] = (df["close"] - df["bb_lower"]) / (df["bb_upper"] - df["bb_lower"])
df["atr_ratio"] = df["atr_14"] / df["close"]
df["hv_regime"] = df["hv_20"] / df["hv_20"].rolling(250, min_periods=250).mean()
df["obv_slope_20"] = (df["obv"] - df["obv"].shift(20)) / (df["vol_sma_20"] * 20)
df["ret_5"] = df["close"] / df["close"].shift(5) - 1.0

CANDIDATES = [
    "sma_gap_20_60", "px_vs_sma60", "rsi_14", "ret_5", "macd_norm",
    "px_vs_sma20", "sma_gap_5_20", "macd_hist_atr", "bb_pctb", "vwap_gap",
    "atr_ratio", "hv_20", "parkinson_20", "bb_bandwidth", "hv_regime",
    "vol_ratio_20", "vol_roc_5", "obv_slope_20",
]

forward = df["fwd_return_5d"].to_numpy(float)
is_big = (np.abs(forward) > 0.01).astype(float)
is_up = np.where(np.abs(forward) > 0.01, (forward > 0).astype(float), np.nan)


def auc(score_values, target):
    """순위 기반 AUC. 0.5 가 '정보 없음' 이다."""
    s = np.asarray(score_values, float)
    t = np.asarray(target, float)
    usable = np.isfinite(s) & np.isfinite(t)
    s, t = s[usable], t[usable]
    ranks = stats.rankdata(s)
    n1 = t.sum()
    n0 = len(t) - n1
    return (ranks[t == 1].sum() - n1 * (n1 + 1) / 2) / (n1 * n0)


records = []
for column in CANDIDATES:
    values = df[column].to_numpy(float)
    usable = np.isfinite(values) & np.isfinite(forward)
    groups = [values[(df["y"] == k).to_numpy() & np.isfinite(values)] for k in (-1, 0, 1)]
    records.append({
        "피처": column,
        "IC": stats.spearmanr(values[usable], forward[usable]).statistic,
        "중립판별 AUC": auc(values, is_big),
        "방향판별 AUC": auc(values, is_up),
        "3분류 KW p": stats.kruskal(*groups).pvalue,
    })

univariate = pd.DataFrame(records)
print(univariate.to_string(index=False, float_format=lambda v: f"{v:11.4f}"))

           피처          IC    중립판별 AUC    방향판별 AUC    3분류 KW p
sma_gap_20_60     -0.0634      0.4647      0.4599      0.0001
  px_vs_sma60     -0.0577      0.4444      0.4648      0.0000
       rsi_14     -0.0459      0.4377      0.4721      0.0000
        ret_5     -0.0427      0.4606      0.4748      0.0004
    macd_norm     -0.0398      0.4464      0.4778      0.0000
  px_vs_sma20     -0.0191      0.4481      0.4906      0.0000
 sma_gap_5_20     -0.0011      0.4488      0.5045      0.0000
macd_hist_atr      0.0134      0.4653      0.5145      0.0047
      bb_pctb     -0.0182      0.4475      0.4924      0.0000
     vwap_gap     -0.0174      0.4488      0.4918      0.0000
    atr_ratio      0.1174      0.5943      0.5617      0.0000
        hv_20      0.1072      0.5835      0.5553      0.0000
 parkinson_20      0.1131      0.5750      0.5568      0.0000
 bb_bandwidth      0.0851      0.5376      0.5348      0.0001
    hv_regime      0.0278      0.5177      0.5131      0.2072
 vol_rat

읽는 법 두 가지입니다.

1. **추세 피처의 IC 가 거의 전부 음수입니다.** 이동평균 위에 있을수록, RSI 가 높을수록
   다음 5거래일 수익률이 **낮았습니다** — 추세 지속이 아니라 **단기 평균회귀**입니다.
   준영님 가설 ① 은 *구분은 되지만 부호가 예상과 반대*입니다.
2. **중립 판별 AUC 상위가 전부 변동성입니다** (`atr_ratio` 0.594). 반면
   **방향 판별 AUC 는 전부 0.46~0.52** 로 정보가 없습니다.
   `vol_ratio_20`·`vol_roc_5` 는 KW p 가 0.6 이상이라 사실상 무정보입니다
   (가설 ④ 의 절반이 여기서 걸립니다).

### 후보 사이 상관 — "방향 피처" 들이 사실상 한 덩어리다

In [8]:
CLUSTER_CHECK = [
    "sma_gap_5_20", "sma_gap_20_60", "px_vs_sma20", "vwap_gap", "rsi_14",
    "bb_pctb", "macd_hist_atr", "atr_ratio", "hv_20", "bb_bandwidth",
    "hv_regime", "vol_ratio_20", "vol_roc_5", "obv_slope_20",
]
correlation = df[CLUSTER_CHECK].corr(method="spearman")

print("|r| >= 0.70 인 쌍")
for i, left in enumerate(CLUSTER_CHECK):
    for right in CLUSTER_CHECK[i + 1:]:
        r = correlation.loc[left, right]
        if abs(r) >= 0.70:
            print(f"  {left:16s} ↔ {right:16s} {r:+.3f}")

|r| >= 0.70 인 쌍
  sma_gap_5_20     ↔ px_vs_sma20      +0.888
  sma_gap_5_20     ↔ vwap_gap         +0.882
  sma_gap_5_20     ↔ rsi_14           +0.832
  sma_gap_5_20     ↔ bb_pctb          +0.806
  sma_gap_5_20     ↔ macd_hist_atr    +0.776
  px_vs_sma20      ↔ vwap_gap         +0.998
  px_vs_sma20      ↔ rsi_14           +0.921
  px_vs_sma20      ↔ bb_pctb          +0.950
  px_vs_sma20      ↔ macd_hist_atr    +0.822
  vwap_gap         ↔ rsi_14           +0.916
  vwap_gap         ↔ bb_pctb          +0.951
  vwap_gap         ↔ macd_hist_atr    +0.829
  rsi_14           ↔ bb_pctb          +0.913
  bb_pctb          ↔ macd_hist_atr    +0.805
  atr_ratio        ↔ hv_20            +0.905
  vol_ratio_20     ↔ vol_roc_5        +0.707


**독립적인 축은 다섯 개뿐입니다** — 단기 위치 · 중기 추세 · 변동성 ·
거래량 크기 · 거래량 방향. `rsi_14` 와 `bb_pctb` 는 r=0.91 이라
둘을 같이 넣으면 기준 ④ 를 정면으로 어깁니다.

그래서 **5~6개가 상한에 가깝고, 8개를 채우려면 억지로 중복을 넣게 됩니다.**

## 6. 문제 분해 — 3분류는 **크기 문제 + 방향 문제**다

중립밴드 ±1.0% 가 5거래일 변동성의 몇 σ 인지 재 봅니다.

In [9]:
sigma5 = np.nanstd(forward)
mean_abs = np.nanmean(np.abs(forward))
print(f"5거래일 수익률 표준편차 σ₅ = {sigma5 * 100:.3f}%")
print(f"E|5거래일 수익률|          = {mean_abs * 100:.3f}%")
print(f"중립밴드 ±1.0% = ±{0.01 / sigma5:.3f} σ₅")
print("  (ADR-AS-0002 는 √5 근사로 ±0.40σ 라 적었다 — 실측과 거의 일치)")
print(f"손익분기 방향정확도(왕복 0.05%) = {(0.5 + 0.0005 / (2 * mean_abs)) * 100:.2f}%")

print("\nhv_20 5분위별 이후 5거래일 라벨 분포")
quintile = pd.qcut(df["hv_20"], 5, labels=["1(최저)", "2", "3", "4", "5(최고)"])
summary = df.groupby(quintile, observed=True).apply(
    lambda g: pd.Series({
        "N": len(g),
        "중립 %": (g["y"] == 0).mean() * 100,
        "상승 %": (g["y"] == 1).mean() * 100,
        "하락 %": (g["y"] == -1).mean() * 100,
        "평균 5일수익 %": g["fwd_return_5d"].mean() * 100,
    }),
    include_groups=False,
)
print(summary.round(2).to_string())

5거래일 수익률 표준편차 σ₅ = 2.422%
E|5거래일 수익률|          = 1.752%
중립밴드 ±1.0% = ±0.413 σ₅
  (ADR-AS-0002 는 √5 근사로 ±0.40σ 라 적었다 — 실측과 거의 일치)
손익분기 방향정확도(왕복 0.05%) = 51.43%

hv_20 5분위별 이후 5거래일 라벨 분포
           N   중립 %   상승 %   하락 %  평균 5일수익 %
hv_20                                       
1(최저)  563.0  47.42  26.29  26.29       0.03
2      563.0  42.63  25.58  31.79      -0.27
3      563.0  42.27  34.28  23.45       0.14
4      563.0  33.57  40.14  26.29       0.36
5(최고)  563.0  27.53  43.69  28.77       0.43


**중립 비율이 47.4% → 27.5% 로 단조 감소합니다. 하락 비율은 26.3% → 28.8% 로 평평합니다.**

변동성은 *"움직일 것인가"* 를 예측하지 *"어느 쪽인가"* 를 예측하지 않습니다.
밴드가 1σ 안쪽이라 **"중립이냐"가 방향 문제가 아니라 크기 문제**가 된 것입니다.

조합 A·B 는 둘 다 크기 질문에 답하는 피처(변동성)는 있었지만,
**방향 질문에 답하는 피처는 아무도 없었습니다.**

## 7. 조합 6안 × 모델 4종 성적표

축을 다르게 조합한 여섯 안을 같은 워크포워드로 돌립니다.

In [10]:
from models.lightgbm import build_lightgbm_baseline
from models.random_forest import build_random_forest_baseline
from models.xgboost import build_xgboost_baseline

MENU = {
    "A 현행": COMBO_A,
    "B 현행": COMBO_B,
    "1안 균형 5": ["sma_gap_20_60", "rsi_14", "macd_hist_atr", "atr_ratio",
                 "obv_slope_20"],
    "2안 변동성강조 6": ["sma_gap_20_60", "rsi_14", "atr_ratio", "bb_bandwidth",
                    "hv_regime", "obv_slope_20"],
    "3안 가설충실 6": ["sma_gap_5_20", "sma_gap_20_60", "macd_hist_atr", "rsi_14",
                   "hv_regime", "vol_ratio_20"],
    "4안 방향만 4": ["sma_gap_20_60", "sma_gap_5_20", "macd_hist_atr", "rsi_14"],
    "5안 변동성만 3": ["atr_ratio", "bb_bandwidth", "hv_regime"],
    "6안 최대 8": ["sma_gap_20_60", "sma_gap_5_20", "rsi_14", "macd_hist_atr",
                 "atr_ratio", "bb_bandwidth", "hv_regime", "obv_slope_20"],
    "참고 원본22": RAW22,
}
BUILDERS = {
    "Logistic": build_logistic_baseline,
    "RandomForest": build_random_forest_baseline,
    "LightGBM": build_lightgbm_baseline,
    "XGBoost": build_xgboost_baseline,
}

ROUND_TRIP_COST = 0.0005  # ADR-AS-0002 주 시나리오: KOSPI200 ETF 왕복 0.05%

results = []
for combo_name, columns in MENU.items():
    for model_name, builder in BUILDERS.items():
        index, prediction = walk_forward(columns, builder)
        truth = df.loc[index, "y"].to_numpy()
        matrix = confusion_matrix(truth, prediction, labels=[-1, 0, 1])
        recall = [matrix[i, i] / max(1, matrix[i].sum()) * 100 for i in range(3)]
        held = index[prediction == 1]
        results.append({
            "조합": combo_name, "피처수": len(columns), "모델": model_name,
            "Acc": accuracy_score(truth, prediction),
            "MacroF1": f1_score(truth, prediction, average="macro"),
            "하락R%": recall[0], "중립R%": recall[1], "상승R%": recall[2],
            "보유일": len(held),
            "비용차감 5일수익%": (forward[held].mean() - ROUND_TRIP_COST) * 100
            if len(held) > 3 else np.nan,
        })

board = pd.DataFrame(results)
pd.set_option("display.width", 200)
print(board.round(4).to_string(index=False))

        조합  피처수           모델    Acc  MacroF1    하락R%    중립R%    상승R%  보유일  비용차감 5일수익%
      A 현행    4     Logistic 0.3958   0.2971  0.0000 63.3987 43.3333  282     -0.1013
      A 현행    4 RandomForest 0.3736   0.3585 23.5294 44.1176 40.9524  256     -0.0020
      A 현행    4     LightGBM 0.3667   0.3471 20.0980 45.7516 39.5238  240      0.1400
      A 현행    4      XGBoost 0.4083   0.3446  6.3725 59.4771 47.1429  281     -0.0319
      B 현행    4     Logistic 0.3792   0.2965  1.9608 58.4967 42.8571  293     -0.0052
      B 현행    4 RandomForest 0.3500   0.3294 20.5882 46.0784 32.8571  242     -0.1723
      B 현행    4     LightGBM 0.3764   0.3531 22.0588 50.3268 34.2857  254     -0.1838
      B 현행    4      XGBoost 0.3722   0.3281 10.7843 52.9412 40.0000  293     -0.2121
   1안 균형 5    5     Logistic 0.3708   0.2810  0.0000 58.4967 41.9048  299     -0.1927
   1안 균형 5    5 RandomForest 0.3861   0.3764 27.9412 42.4837 43.3333  250      0.1903
   1안 균형 5    5     LightGBM 0.3722   0.3491 17.6471 4

In [11]:
print("조합별 4모델 평균 — 모델을 바꿔도 조합의 성격이 유지되는가")
grouped = board.groupby("조합", sort=False).agg(
    피처수=("피처수", "first"),
    Acc평균=("Acc", "mean"),
    MacroF1평균=("MacroF1", "mean"),
    하락R평균=("하락R%", "mean"),
    비용차감수익평균=("비용차감 5일수익%", "mean"),
)
print(grouped.round(4).to_string())

조합별 4모델 평균 — 모델을 바꿔도 조합의 성격이 유지되는가
            피처수   Acc평균  MacroF1평균    하락R평균  비용차감수익평균
조합                                                   
A 현행          4  0.3861     0.3369  12.5000    0.0012
B 현행          4  0.3694     0.3268  13.8480   -0.1434
1안 균형 5       5  0.3767     0.3330  13.6029   -0.0249
2안 변동성강조 6    6  0.3816     0.3525  18.7500    0.1470
3안 가설충실 6     6  0.3431     0.3162  16.7892    0.0103
4안 방향만 4      4  0.3490     0.3181  15.6863   -0.0361
5안 변동성만 3     3  0.3899     0.3504  24.2647   -0.0038
6안 최대 8       8  0.3847     0.3592  22.0588    0.1839
참고 원본22      22  0.3622     0.3467  27.5735    0.0626


## 8. 하락 예측이 안 나온 진짜 이유

### 8-1. 피처를 한 줄도 안 바꾸고 **모델만** 바꿔 본다

In [12]:
print("조합 A — 피처 동일, 모델만 교체")
for model_name, builder in BUILDERS.items():
    score(f"  {model_name}", *walk_forward(COMBO_A, builder))

조합 A — 피처 동일, 모델만 교체
  Logistic               Acc=0.3958 MacroF1=0.2971 Recall(하/중/상)=  0.0/ 63.4/ 43.3% 하락예측=  0


  RandomForest           Acc=0.3736 MacroF1=0.3585 Recall(하/중/상)= 23.5/ 44.1/ 41.0% 하락예측=164


  LightGBM               Acc=0.3667 MacroF1=0.3471 Recall(하/중/상)= 20.1/ 45.8/ 39.5% 하락예측=159


  XGBoost                Acc=0.4083 MacroF1=0.3446 Recall(하/중/상)=  6.4/ 59.5/ 47.1% 하락예측= 62


**같은 피처인데 하락 Recall 이 0.0% → 23.5% 입니다.**

즉 *"하락 예측 0개"* 는 피처가 방향 정보를 못 담아서가 아니라,
**로지스틱이 클래스 사전확률(중립 42.5%)에 눌려 소수 클래스를 아예 안 부르기 때문**입니다.

### 8-2. `class_weight="balanced"` 한 줄

In [13]:
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler


def build_balanced():
    """준영님 build_logistic_baseline 과 같되 class_weight 만 켠 것."""
    return Pipeline([
        ("scaler", StandardScaler()),
        ("classifier", LogisticRegression(
            random_state=42, max_iter=1000, class_weight="balanced",
        )),
    ])


for tag, columns in (("조합 A", COMBO_A), ("조합 B", COMBO_B),
                     ("2안", MENU["2안 변동성강조 6"])):
    score(f"{tag} (기본)", *walk_forward(columns))
    score(f"{tag} + balanced", *walk_forward(columns, build_balanced))
    print()

조합 A (기본)                Acc=0.3958 MacroF1=0.2971 Recall(하/중/상)=  0.0/ 63.4/ 43.3% 하락예측=  0
조합 A + balanced          Acc=0.4028 MacroF1=0.3570 Recall(하/중/상)= 11.8/ 54.9/ 46.7% 하락예측= 63



조합 B (기본)                Acc=0.3792 MacroF1=0.2965 Recall(하/중/상)=  2.0/ 58.5/ 42.9% 하락예측=  8
조합 B + balanced          Acc=0.3792 MacroF1=0.3258 Recall(하/중/상)=  7.8/ 53.6/ 44.3% 하락예측= 54



2안 (기본)                  Acc=0.3917 MacroF1=0.3362 Recall(하/중/상)= 12.7/ 63.1/ 30.0% 하락예측= 81
2안 + balanced            Acc=0.3806 MacroF1=0.3565 Recall(하/중/상)= 22.1/ 51.0/ 34.8% 하락예측=160



조합 A 는 **Accuracy 와 Macro F1 이 둘 다 올라갑니다** (0.3958→0.4028, 0.2971→0.3570).
보통 balanced 는 정확도를 내주고 소수 클래스를 얻는 거래인데,
여기서는 그 거래조차 없었습니다 — **하락을 아예 안 부르는 것이 정확도에도 손해**였습니다.

### 8-3. 그런데 — 하락 Recall 은 **실행상 성과에 영향이 없다**

[ADR-AS-0002](../../docs/decisions/0002-예측대상과-레이블.md) 가 포지션을 `{0, +1}` 로
못 박았습니다. 공매도가 없으므로 **하락 예측은 "보유하지 않음(현금)"으로만 실행**됩니다.

하락으로 부르든 중립으로 부르든 그날 포지션은 똑같이 0 입니다. 확인합니다.

In [14]:
for tag, columns in (("조합 A", COMBO_A), ("조합 B", COMBO_B)):
    index, prediction = walk_forward(columns)
    for label, adjusted in (("현행 3분류", prediction),
                            ("하락→중립 병합", np.where(prediction == -1, 0, prediction))):
        held = adjusted == 1
        returns = forward[index][held]
        print(f"  {tag} · {label:14s} 보유일수={int(held.sum()):3d} "
              f"평균 5일수익={returns.mean() * 100:+.4f}%")
    print()

  조합 A · 현행 3분류         보유일수=282 평균 5일수익=-0.0513%


  조합 A · 하락→중립 병합       보유일수=282 평균 5일수익=-0.0513%

  조합 B · 현행 3분류         보유일수=293 평균 5일수익=+0.0448%
  조합 B · 하락→중립 병합       보유일수=293 평균 5일수익=+0.0448%



**완전히 동일합니다.** 하락 Recall 을 1.96% 에서 20% 로 올려도
매매 결과는 한 줄도 바뀌지 않습니다.

그러므로 *"하락·중립·상승을 모두 예측하는지"* 는 **리포트의 진단 지표로는 유효하지만,
피처 선정의 합격 조건으로 쓰면 안 됩니다.** 실행상 중요한 구분은
**"상승이냐 아니냐"** 뿐입니다.

## 9. 유효표본과 유의선 — 조합 A·B 의 차이는 잴 수 없다

매일 5거래일 앞을 보는 라벨은 이웃 관측이 4일을 공유합니다.
`evaluation/horizon.py::overlap_vif(5)` 가 그 분산팽창계수를 계산합니다.

In [15]:
from evaluation.horizon import overlap_vif

vif = overlap_vif(5)
baseline = max(np.mean(y_valid_all == c) for c in (-1, 0, 1))
effective = len(y_valid_all) / vif
standard_error = np.sqrt(baseline * (1 - baseline) / effective)

print(f"검증 {len(y_valid_all)}행 ÷ VIF {vif:.2f} = 유효표본 {effective:.0f}행")
print(f"최빈 기준선(항상 중립) = {baseline * 100:.2f}%")
print(f"정확도 표준오차 = ±{standard_error * 100:.2f}%p")
print()
print(f"  단일 검정 유의선(단측 5%)      = {(baseline + 1.645 * standard_error) * 100:.2f}%")
print(f"  9개 조합 본페로니 보정 유의선   = {(baseline + 2.54 * standard_error) * 100:.2f}%")
print(f"  36개(조합×모델) 보정 유의선     = {(baseline + 3.00 * standard_error) * 100:.2f}%")
print()
best = board.loc[board["Acc"].idxmax()]
print(f"이 노트북의 최고 성적 = {best['조합']} / {best['모델']} : {best['Acc'] * 100:.2f}%")
print(f"조합 A(39.58%) 와 조합 B(37.92%) 의 차이 1.66%p 는 "
      f"표준오차 ±{standard_error * 100:.2f}%p 안이다 — 우열을 가릴 수 없다.")

검증 720행 ÷ VIF 3.78 = 유효표본 190행
최빈 기준선(항상 중립) = 42.50%
정확도 표준오차 = ±3.58%p

  단일 검정 유의선(단측 5%)      = 48.39%
  9개 조합 본페로니 보정 유의선   = 51.60%
  36개(조합×모델) 보정 유의선     = 53.25%

이 노트북의 최고 성적 = 5안 변동성만 3 / Logistic : 41.11%
조합 A(39.58%) 와 조합 B(37.92%) 의 차이 1.66%p 는 표준오차 ±3.58%p 안이다 — 우열을 가릴 수 없다.


## 정리

| 준영님 기준·가설 | 판정 |
|---|---|
| 기준 ④ 중복 금지 | 🔴 `bb_mid ≡ sma_20`, `macd ≡ ema_12 − ema_26` 등 **정의상 같은 열**이 있다 |
| 기준 ⑤ 미래 데이터 금지 | ✅ 통과. 다만 **수준값 10개**가 학습 범위 밖으로 나간다 (트리에서 특히 위험) |
| 기준 ⑥ 5~8개 | ⚠️ 독립적인 축이 **다섯 개**뿐이라 5~6개가 상한 |
| 가설 ① 이동평균·MACD 로 방향 | ⚠️ 구분은 되지만 **부호가 반대** (평균회귀) |
| 가설 ② RSI·볼린저 위치 | 🔴 둘이 r=0.91 — **하나만** |
| 가설 ③ 변동성 → 유지/움직임 | ✅ **정확** (중립 47.4%→27.5% 단조 감소) |
| 가설 ④ 거래량 확인 | ⚠️ `vol_ratio_20` 은 무정보(p=0.60), `obv` 기울기만 살아남음 |
| 기준 ⑦ 세 클래스 모두 예측 | 🔴 원인은 피처가 아니라 **클래스 쏠림**. 그리고 실행상 하락은 중립과 같다 |

**다음에 무엇을 하면 되는지**는 결론 문서 §6 에 네 가지로 정리했습니다 →
[`docs/모델파트/version1.0/X조합-피처선정-검토.md`](../../docs/모델파트/version1.0/X조합-피처선정-검토.md)